# MuleNet MVP — Exploration & Walkthrough

Quick EDA + end-to-end walkthrough of the mule account classifier.

**Target variable:** `F3924`

**Bank-highlighted features:** F115, F321, F527, F531, F670, F1692, F2082, F2122, F2582, F2678, F2737, F2956, F3043, F3836, F3887, F3889, F3891, F3894

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

TARGET_COL = 'F3924'
DATA_PATH = '../data/dataset.csv'  # update with your actual filename

df = pd.read_csv(DATA_PATH)
print(df.shape)
df.head()

## 1. Target distribution

In [ ]:
print(df[TARGET_COL].value_counts())
print(df[TARGET_COL].value_counts(normalize=True))

df[TARGET_COL].value_counts().plot(kind='bar', title='Target distribution (F3924)')
plt.show()

## 2. Missing values overview

In [ ]:
missing = df.isna().mean().sort_values(ascending=False)
print('Columns with >50% missing:', (missing > 0.5).sum())
missing.head(20)

## 3. Distribution of bank-highlighted features by target class

In [ ]:
HINT_FEATURES = ['F115','F321','F527','F531','F670','F1692','F2082','F2122',
                 'F2582','F2678','F2737','F2956','F3043','F3836','F3887',
                 'F3889','F3891','F3894']

available = [c for c in HINT_FEATURES if c in df.columns]
print(f'{len(available)}/{len(HINT_FEATURES)} hint features present')

fig, axes = plt.subplots(3, 3, figsize=(14, 10))
for ax, col in zip(axes.flatten(), available[:9]):
    for cls in df[TARGET_COL].unique():
        subset = df[df[TARGET_COL] == cls][col]
        ax.hist(pd.to_numeric(subset, errors='coerce').dropna(), bins=30, alpha=0.5, label=f'class {cls}')
    ax.set_title(col)
    ax.legend()
plt.tight_layout()
plt.show()

## 4. Run the full training pipeline

This calls the same pipeline as `src/train.py` — run from the command line for the full run:

```bash
python ../src/train.py --data ../data/dataset.csv
```

Or import and run inline below.

In [ ]:
import sys
sys.path.append('../src')
from train import basic_clean, engineer_features, train_model, evaluate
from sklearn.model_selection import train_test_split

df_clean = basic_clean(df)
df_feat = engineer_features(df_clean)

y = df_feat[TARGET_COL].astype(int)
X = df_feat.drop(columns=[TARGET_COL])

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

model = train_model(X_train, y_train, X_val, y_val)
metrics, proba = evaluate(model, X_val, y_val)

## 5. SHAP explainability — example

In [ ]:
import shap
explainer = shap.TreeExplainer(model)
sample = X_val.sample(min(300, len(X_val)), random_state=42)
shap_values = explainer.shap_values(sample)
shap.summary_plot(shap_values, sample, max_display=15)

## 6. Next steps

- Tune hyperparameters with cross-validation
- Add domain-meaningful feature engineering once column semantics are confirmed
- Build the GraphSAGE network layer (see `docs/roadmap.md`)